In [8]:
import sys, os

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(_notebook_dir, '..', 'scripts'))

import pandas as pd
from data_utils import PARQUET_RAW as PARQUET, REF

print("PARQUET path:", PARQUET)
print("REF path:", REF)

PARQUET path: c:\Disertation\UoB-GeneTraceAI-25-26\data\parquet/raw_data
REF path: c:\Disertation\UoB-GeneTraceAI-25-26\reference


In [9]:
sample_info = pd.read_parquet(os.path.join(PARQUET, '9_DepMap_sample_info.parquet'))
print('sample_info shape:', sample_info.shape)
print('columns:', sample_info.columns.tolist())
print(sample_info.head(3))

sample_info shape: (1840, 29)
columns: ['DepMap_ID', 'cell_line_name', 'stripped_cell_line_name', 'CCLE_Name', 'alias', 'COSMICID', 'sex', 'source', 'RRID', 'WTSI_Master_Cell_ID', 'sample_collection_site', 'primary_or_metastasis', 'primary_disease', 'Subtype', 'age', 'Sanger_Model_ID', 'depmap_public_comments', 'lineage', 'lineage_subtype', 'lineage_sub_subtype', 'lineage_molecular_subtype', 'default_growth_pattern', 'model_manipulation', 'model_manipulation_details', 'patient_id', 'parent_depmap_id', 'Cellosaurus_NCIt_disease', 'Cellosaurus_NCIt_id', 'Cellosaurus_issues']
    DepMap_ID cell_line_name stripped_cell_line_name  \
0  ACH-000016         SLR 21                   SLR21   
1  ACH-000032     MHH-CALL-3                MHHCALL3   
2  ACH-000033      NCI-H1819                NCIH1819   

                                     CCLE_Name alias  COSMICID     sex  \
0                                 SLR21_KIDNEY   NaN       NaN     NaN   
1  MHHCALL3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE 

In [10]:
cellosaurus = pd.read_parquet(os.path.join(PARQUET, '7_cellosaurus.parquet'))
print('cellosaurus shape:', cellosaurus.shape)
print('columns:', cellosaurus.columns.tolist())
print(cellosaurus.head(3))

cellosaurus shape: (152231, 17)
columns: ['Identifier (cell line name)', 'Accession (CVCL_xxxx)', 'Secondary accession number(s)', 'Synonyms', 'Cross-references', 'References identifiers', 'Web pages', 'Comments', 'STR profile data', 'Diseases', 'Species of origin', 'Hierarchy', 'Originate from same individual', 'Sex of cell', 'Age of donor at sampling', 'Category', 'Date (entry history)']
  Identifier (cell line name) Accession (CVCL_xxxx)  \
0            #132 PC3-1-SC-E8             CVCL_B0T9   
1             #132 PL12 SC-D1             CVCL_B0T8   
2                   #15310-LN             CVCL_E548   

  Secondary accession number(s)  \
0                           NaN   
1                           NaN   
2                           NaN   

                                            Synonyms  \
0                                         Z48-5MG-70   
1                                         Z48-5MG-63   
2  15310-LN; TER461; TER-461; Ter 461; TER479; TE...   

                    

In [11]:
# Step 1: Take only needed columns from sample_info
cell_line_lookup = sample_info[[
    "DepMap_ID",
    "cell_line_name",
    "stripped_cell_line_name",
    "RRID"
]].copy()

# Step 2: Rename to contract spec
cell_line_lookup = cell_line_lookup.rename(columns={
    "DepMap_ID": "model_id",
    "RRID":      "rrid"
})

# Step 3: Slim down cellosaurus to only what we need
cello_slim = cellosaurus[[
    "Accession (CVCL_xxxx)",
    "Synonyms"
]].rename(columns={
    "Accession (CVCL_xxxx)": "cvcl_accession",
    "Synonyms":               "synonyms"
})

# Step 4: Left join — sample_info stays at 1,840 rows, cellosaurus adds columns
cell_line_lookup = cell_line_lookup.merge(
    cello_slim,
    left_on="rrid",
    right_on="cvcl_accession",
    how="left"
)

# Step 5: Convert semicolon-delimited synonyms → pipe-delimited
cell_line_lookup["synonyms"] = (
    cell_line_lookup["synonyms"]
    .fillna("")
    .str.strip()
    .str.replace(r";\s*", "|", regex=True)
    .replace("", float("nan"))
)

# Step 6: Force plain object dtype
cell_line_lookup = cell_line_lookup.astype(object)

# Step 7: Final column order per contract
cell_line_lookup = cell_line_lookup[[
    "model_id", "cell_line_name", "stripped_cell_line_name",
    "cvcl_accession", "synonyms", "rrid"
]]

print(cell_line_lookup.shape)
print(cell_line_lookup.head(5))

(1840, 6)
     model_id cell_line_name stripped_cell_line_name cvcl_accession  \
0  ACH-000016         SLR 21                   SLR21      CVCL_V607   
1  ACH-000032     MHH-CALL-3                MHHCALL3      CVCL_0089   
2  ACH-000033      NCI-H1819                NCIH1819      CVCL_1497   
3  ACH-000043       Hs 895.T                  HS895T      CVCL_0993   
4  ACH-000049         HEK TE                   HEKTE      CVCL_WS59   

                                            synonyms       rrid  
0                                             SLR 21  CVCL_V607  
1  Mhh-Call 3|MHH cALL 3|MHH-CALL3|MHH-cALL3|MHHC...  CVCL_0089  
2                              H1819|H-1819|NCIH1819  CVCL_1497  
3            Hs-895-T|Hs 895 T|Hs895.T|Hs895T|HS895T  CVCL_0993  
4                                       HEK-TE|HEKTE  CVCL_WS59  


In [12]:
print("=== VALIDATION ===")
print(f"Total rows:                    {len(cell_line_lookup):,}")
print(f"Null model_id:                 {cell_line_lookup['model_id'].isna().sum()}")
print(f"Duplicate model_id:            {cell_line_lookup['model_id'].duplicated().sum()}")
print(f"Null cell_line_name:           {cell_line_lookup['cell_line_name'].isna().sum()}")
print(f"Has cvcl_accession:            {cell_line_lookup['cvcl_accession'].notna().sum():,}")
print(f"Has synonyms:                  {cell_line_lookup['synonyms'].notna().sum():,}")
print(f"No cellosaurus match:          {cell_line_lookup['cvcl_accession'].isna().sum():,}")

=== VALIDATION ===
Total rows:                    1,840
Null model_id:                 0
Duplicate model_id:            0
Null cell_line_name:           92
Has cvcl_accession:            1,816
Has synonyms:                  1,684
No cellosaurus match:          24


In [13]:
OUT = os.path.join(REF, "cell_line_lookup.parquet")

cell_line_lookup.to_parquet(OUT, index=False, engine="fastparquet")

confirm = pd.read_parquet(OUT, engine="fastparquet")
print(f"Saved and verified: {confirm.shape}")
print(f"File size: {os.path.getsize(OUT):,} bytes")

Saved and verified: (1840, 6)
File size: 90,772 bytes
